# Practical 2: Tokenization and Normalization

**Concept:** Before a machine can understand text, we must decide exactly what counts as a single meaningful unit. **Tokenization** is the process of splitting text into these units (tokens — usually words). **Normalization** is the process of cleaning and standardizing tokens so that different surface forms of the same word are treated alike.

In this notebook we explore:
1. Sentence Tokenization
2. Word Tokenization
3. Different Tokenizers (Whitespace, WordPunct, Treebank, Regexp)
4. Normalization (lowercasing, removing noise, Unicode normalization)
5. Stemming vs Lemmatization

## Import Required Libraries

In [1]:
import nltk
import re
import string
import unicodedata

# NLTK tokenizers
from nltk.tokenize import (
    sent_tokenize,
    word_tokenize,
    WhitespaceTokenizer,
    WordPunctTokenizer,
    TreebankWordTokenizer,
    RegexpTokenizer,
)

# Stopwords, stemming, lemmatization
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

## Download Required NLTK Resources
We additionally need the **averaged_perceptron_tagger** for POS-aware lemmatization at the end of this notebook.

In [2]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")

[nltk_data] Downloading package punkt to /home/tejas/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/tejas/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/tejas/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/tejas/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/tejas/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

## Create the Corpus
Our sample text intentionally includes uppercase words, contractions, numbers, URLs, emoticons and Unicode characters so that every tokenizer behaves differently.

In [3]:
text = """
Natural Language Processing (NLP) enables machines to understand human language.
Visit https://example.com or email support@example.com for more details.
The weather in Madrid is 25°C today. We've studied NLP since 2020!
Happily, running and running are handled differently by stemming & lemmatization.
Café (with accent) and Cafe (without accent) look the same to a human. :) 
"""

## 1. Sentence Tokenization
`sent_tokenize` splits the raw text into sentences. It uses an unsupervised model that has learned sentence boundaries from real text.

In [4]:
# Sentence tokenization
sentences = sent_tokenize(text)

print("Sentence Tokenization")
print("-" * 50)

for i, sentence in enumerate(sentences, start=1):
    print(f"Sentence {i}: {sentence.strip()}")

Sentence Tokenization
--------------------------------------------------
Sentence 1: Natural Language Processing (NLP) enables machines to understand human language.
Sentence 2: Visit https://example.com or email support@example.com for more details.
Sentence 3: The weather in Madrid is 25°C today.
Sentence 4: We've studied NLP since 2020!
Sentence 5: Happily, running and running are handled differently by stemming & lemmatization.
Sentence 6: Café (with accent) and Cafe (without accent) look the same to a human.
Sentence 7: :)


## 2. Word Tokenization
`word_tokenize` splits a sentence into word tokens. Notice that punctuation is kept as its own token, which is useful for many tasks.

In [5]:
# Word tokenization on the first sentence
tokens = word_tokenize(sentences[0])

print("Word Tokenization")
print("-" * 50)
print(tokens)

Word Tokenization
--------------------------------------------------
['Natural', 'Language', 'Processing', '(', 'NLP', ')', 'enables', 'machines', 'to', 'understand', 'human', 'language', '.']


## 3. Comparing Different Tokenizers
Different tokenizers make different trade-offs. We tokenize the **same sentence** with four NLTK tokenizers and compare the results.

In [6]:
# The test sentence contains contractions, punctuation and a URL
test_sentence = "We've visited https://example.com, it's great!"

In [7]:
# WhitespaceTokenizer: only splits on spaces
ws_tokens = WhitespaceTokenizer().tokenize(test_sentence)
print("WhitespaceTokenizer")
print(ws_tokens)
print()

WhitespaceTokenizer
["We've", 'visited', 'https://example.com,', "it's", 'great!']



In [8]:
# WordPunctTokenizer: splits words AND punctuation into separate tokens
wp_tokens = WordPunctTokenizer().tokenize(test_sentence)
print("WordPunctTokenizer")
print(wp_tokens)
print()

WordPunctTokenizer
['We', "'", 've', 'visited', 'https', '://', 'example', '.', 'com', ',', 'it', "'", 's', 'great', '!']



In [9]:
# TreebankWordTokenizer: handles contractions smartly (keeps the apostrophe attach)
tb_tokens = TreebankWordTokenizer().tokenize(test_sentence)
print("TreebankWordTokenizer")
print(tb_tokens)
print()

TreebankWordTokenizer
['We', "'ve", 'visited', 'https', ':', '//example.com', ',', 'it', "'s", 'great', '!']



In [10]:
# RegexpTokenizer: uses a regex pattern to decide what a token is
# Here we only keep alphabetic words and numbers (2+ chars), dropping punctuation
re_tokens = RegexpTokenizer(r"\w+").tokenize(test_sentence)
print("RegexpTokenizer (\\w+)")
print(re_tokens)

RegexpTokenizer (\w+)
['We', 've', 'visited', 'https', 'example', 'com', 'it', 's', 'great']


## 4. Normalization
**Normalization** brings all tokens into a standard form so that equivalent words map to the same representation.

### 4.1 Lowercasing
Lowercasing collapses "NLP" and "nlp" into one token, reducing vocabulary size.

In [11]:
# Lowercase every token
lower_tokens = [token.lower() for token in re_tokens]

print("Lowercased Tokens")
print("-" * 50)
print(lower_tokens)

Lowercased Tokens
--------------------------------------------------
['we', 've', 'visited', 'https', 'example', 'com', 'it', 's', 'great']


### 4.2 Removing Noise (URLs, Emails, Mentions)
Regular expressions let us strip out URLs, emails and social-media handles which add noise to most models.

In [12]:
# Remove URLs
clean_text = re.sub(r"https?://\S+|www\.\S+", "", text)

# Remove email addresses
clean_text = re.sub(r"\S+@\S+", "", clean_text)

print("Text After Removing Noise")
print("-" * 50)
print(clean_text.strip())

Text After Removing Noise
--------------------------------------------------
Natural Language Processing (NLP) enables machines to understand human language.
Visit  or email  for more details.
The weather in Madrid is 25°C today. We've studied NLP since 2020!
Happily, running and running are handled differently by stemming & lemmatization.
Café (with accent) and Cafe (without accent) look the same to a human. :)


### 4.3 Collapsing Whitespace
Multiple spaces, tabs and newlines are collapsed into a single space so the text is a clean stream of words.

In [13]:
# Replace any sequence of whitespace with a single space
normalized = re.sub(r"\s+", " ", clean_text).strip()

print("Text After Collapsing Whitespace")
print("-" * 50)
print(normalized)

Text After Collapsing Whitespace
--------------------------------------------------
Natural Language Processing (NLP) enables machines to understand human language. Visit or email for more details. The weather in Madrid is 25°C today. We've studied NLP since 2020! Happily, running and running are handled differently by stemming & lemmatization. Café (with accent) and Cafe (without accent) look the same to a human. :)


### 4.4 Unicode Normalization
Accented characters can be written in different byte forms. NFKD normalization decomposes them, letting us remove the accents so "Café" and "Cafe" become identical.

In [14]:
# Normalize Unicode (NFKD) then strip combining accent marks
def strip_accents(s):
    decomposed = unicodedata.normalize("NFKD", s)
    return "".join(
        char for char in decomposed
        if not unicodedata.combining(char)
    )

sample = "Café vs Cafe, Naïve vs Naive"
print("Original:", sample)
print("Accent Stripped:", strip_accents(sample))

Original: Café vs Cafe, Naïve vs Naive
Accent Stripped: Cafe vs Cafe, Naive vs Naive


### 4.5 Full Normalization Pipeline
Putting it all together into one reusable function that tokenizes and normalizes raw text.

In [15]:
def normalize_text(raw_text):
    """Tokenize and fully normalize a raw text."""
    # Remove URLs and emails
    text_clean = re.sub(r"https?://\S+|www\.\S+", "", raw_text)
    text_clean = re.sub(r"\S+@\S+", "", text_clean)

    # Collapse whitespace and strip accents
    text_clean = re.sub(r"\s+", " ", text_clean).strip()
    text_clean = strip_accents(text_clean)

    # Tokenize keeping only words and numbers
    tokens = RegexpTokenizer(r"\w+").tokenize(text_clean.lower())

    # Remove stopwords
    stop_words = set(stopwords.words("english"))
    tokens = [tok for tok in tokens if tok not in stop_words]

    return tokens


normalized_tokens = normalize_text(text)

print("Fully Normalized Tokens")
print("-" * 50)
print(normalized_tokens)

Fully Normalized Tokens
--------------------------------------------------
['natural', 'language', 'processing', 'nlp', 'enables', 'machines', 'understand', 'human', 'language', 'visit', 'email', 'details', 'weather', 'madrid', '25', 'c', 'today', 'studied', 'nlp', 'since', '2020', 'happily', 'running', 'running', 'handled', 'differently', 'stemming', 'lemmatization', 'cafe', 'accent', 'cafe', 'without', 'accent', 'look', 'human']


## 5. Stemming vs Lemmatization
**Stemming** chops off word endings using rules (fast, but can produce non-words). **Lemmatization** uses a dictionary + POS tag to return the real base form (slower, more accurate).

In [16]:
# Sample words that behave differently
sample_words = ["running", "studies", "better", "caring", "geese"]

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print(f"{'Word':<10}{'Stem':<12}{'Lemma'}")
print("-" * 32)

for word in sample_words:
    print(f"{word:<10}{stemmer.stem(word):<12}{lemmatizer.lemmatize(word)}")

Word      Stem        Lemma
--------------------------------
running   run         running
studies   studi       study
better    better      better
caring    care        caring
geese     gees        goose


### 5.1 POS-aware Lemmatization
Lemmatization works better when it knows the Part-of-Speech. For example, "better" as an **adjective** should stay "better" and not become "bet". We pass the POS tag to the lemmatizer.

In [17]:
# Map NLTK POS tags to WordNet POS tags (a -> adjective, v -> verb, r -> adverb, n -> noun)
def wn_tag(nltk_tag):
    if nltk_tag.startswith("J"):
        return "a"
    elif nltk_tag.startswith("V"):
        return "v"
    elif nltk_tag.startswith("R"):
        return "r"
    else:
        return "n"

# Example sentence
sample_sentence = "The leaves fell better when they were running."

# Get (word, POS) pairs
tagged = pos_tag(word_tokenize(sample_sentence))

print("POS-aware Lemmatization")
print("-" * 50)

for word, tag in tagged:
    lemma = lemmatizer.lemmatize(word, wn_tag(tag))
    print(f"{word:<10} {tag:<6} -> {lemma}")

POS-aware Lemmatization
--------------------------------------------------
The        DT     -> The
leaves     NNS    -> leaf
fell       VBD    -> fell
better     RB     -> well
when       WRB    -> when
they       PRP    -> they
were       VBD    -> be
running    VBG    -> run
.          .      -> .


## Summary

| Step | Purpose | Result |
|------|---------|--------|
| Sentence Tokenization | Split text into sentences | 6 sentences |
| Word Tokenization | Split sentences into tokens | punctuation kept separate |
| Tokenizer Choice | Trade-off between speed & precision | Whitespace < WordPunct < Treebank < Regexp |
| Lowercasing | Collapse case variants | smaller vocabulary |
| Noise Removal | Remove URLs/emails | cleaner text |
| Whitespace Collapse | Standardize spacing | single-space stream |
| Unicode Normalization | Decompose accents | Café → Cafe |
| Stemming | Chop affixes by rules | fast but non-words |
| Lemmatization | Dictionary + POS base form | accurate real words |

This notebook produced a reusable `normalize_text()` function which will be reused in the upcoming practicals for building TF-IDF, Word2Vec and classification models.